# Reto Integrador - Unidad 1

## Parte 3. Ingeniería de Datos con Pandas

En este notebook se realiza la carga, limpieza, transformación y exportación del conjunto de datos utilizando Pandas y SQLite.

## - Se Importan librerías

In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

## - Se carga el dataset

In [4]:
df = pd.read_csv(
    "../data/ventas_techventas_limpio.csv",
    encoding="latin-1",
    parse_dates=["fecha"]
)

df.head(60)

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2,1200.0,0.10,Ana Lopez
1,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15,25.0,0.00,Carlos Ruiz
2,1003,2024-01-10,C003,DataSolutions,Centro,Monitor 27,Electronica,5,350.0,0.05,Ana Lopez
3,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10,85.0,0.10,Luis Mora
4,1005,2024-01-15,C004,CloudNet,Este,Laptop Pro 15,Electronica,3,1200.0,0.15,Carlos Ruiz
5,1006,2024-01-18,C005,Sistemas Globales,Oeste,SSD 1TB,Almacenamiento,20,95.0,0.00,Maria Torres
6,1007,2024-01-20,C002,Innovatek,Sur,Monitor 27,Electronica,2,350.0,0.05,Ana Lopez
7,1008,2024-01-22,C006,NetPrime,Norte,Router WiFi 6,Redes,8,120.0,0.00,Luis Mora
8,1009,2024-01-25,C003,DataSolutions,Centro,SSD 1TB,Almacenamiento,12,95.0,0.10,Maria Torres
9,1010,2024-01-28,C007,AlphaTech,Sur,Laptop Pro 15,Electronica,1,1200.0,0.00,Carlos Ruiz


# 3.1 Carga y exploración del conjunto de datos

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_id         60 non-null     int64         
 1   fecha            60 non-null     datetime64[us]
 2   cliente_id       60 non-null     str           
 3   cliente_nombre   60 non-null     str           
 4   region           60 non-null     str           
 5   producto         59 non-null     str           
 6   categoria        60 non-null     str           
 7   cantidad         60 non-null     int64         
 8   precio_unitario  60 non-null     float64       
 9   descuento        60 non-null     float64       
 10  vendedor         60 non-null     str           
dtypes: datetime64[us](1), float64(2), int64(2), str(6)
memory usage: 5.3 KB


### Observaciones

- El conjunto de datos contiene 60 registros y 11 columnas.

- La columna **fecha** fue cargada correctamente como tipo fecha gracias al parámetro `parse_dates`.

- Se observa un único valor nulo en la columna **producto**, además de un registro con una cantidad negativa, los cuales deberán ser tratados durante la etapa de limpieza.

## 3.2 Limpieza del conjunto de datos

In [6]:
df.isna().sum()

order_id           0
fecha              0
cliente_id         0
cliente_nombre     0
region             0
producto           1
categoria          0
cantidad           0
precio_unitario    0
descuento          0
vendedor           0
dtype: int64

## - Se eliminan los registros

In [7]:
filas_iniciales = len(df)

df = df[df["producto"].notna()]
df = df[df["producto"] != ""]
df = df[df["cantidad"] > 0]

filas_finales = len(df)

print("Registros iniciales:", filas_iniciales)
print("Registros finales:", filas_finales)
print("Registros eliminados:", filas_iniciales - filas_finales)

Registros iniciales: 60
Registros finales: 58
Registros eliminados: 2


### Justificación

- Se eliminaron los registros que presentaban un producto vacío y aquellos con una cantidad menor o igual a cero, ya que representan datos inconsistentes que podrían afectar los resultados del análisis.

- En total se eliminaron **2 registros**.

## 3.3 Creación de nuevas variables

In [9]:
df["ingreso_bruto"] = df["cantidad"] * df["precio_unitario"]

df["ingreso_neto"] = df["ingreso_bruto"] * (1 - df["descuento"])

df["mes"] = df["fecha"].dt.to_period("M")

df.head(60)

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor,ingreso_bruto,ingreso_neto,mes
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2,1200.0,0.10,Ana Lopez,2400.0,2160.00,2024-01
1,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15,25.0,0.00,Carlos Ruiz,375.0,375.00,2024-01
2,1003,2024-01-10,C003,DataSolutions,Centro,Monitor 27,Electronica,5,350.0,0.05,Ana Lopez,1750.0,1662.50,2024-01
3,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10,85.0,0.10,Luis Mora,850.0,765.00,2024-01
4,1005,2024-01-15,C004,CloudNet,Este,Laptop Pro 15,Electronica,3,1200.0,0.15,Carlos Ruiz,3600.0,3060.00,2024-01
5,1006,2024-01-18,C005,Sistemas Globales,Oeste,SSD 1TB,Almacenamiento,20,95.0,0.00,Maria Torres,1900.0,1900.00,2024-01
6,1007,2024-01-20,C002,Innovatek,Sur,Monitor 27,Electronica,2,350.0,0.05,Ana Lopez,700.0,665.00,2024-01
7,1008,2024-01-22,C006,NetPrime,Norte,Router WiFi 6,Redes,8,120.0,0.00,Luis Mora,960.0,960.00,2024-01
8,1009,2024-01-25,C003,DataSolutions,Centro,SSD 1TB,Almacenamiento,12,95.0,0.10,Maria Torres,1140.0,1026.00,2024-01
9,1010,2024-01-28,C007,AlphaTech,Sur,Laptop Pro 15,Electronica,1,1200.0,0.00,Carlos Ruiz,1200.0,1200.00,2024-01


## Resumen mensual

In [10]:
resumen = df.groupby("mes").agg(
    total=("ingreso_neto", "sum"),
    numero_ventas=("order_id", "count")
)

resumen

,total,numero_ventas
mes,,
2024-01,13773.50,10
2024-02,13050.00,10
2024-03,14454.00,11
2024-04,16689.00,11
2024-05,11922.25,9
2024-06,11209.25,7


## - Se crea la tabla de metas

In [11]:
metas = pd.DataFrame({
    "mes": [
        "2024-01",
        "2024-02",
        "2024-03",
        "2024-04",
        "2024-05",
        "2024-06"
    ],
    "meta": [
        15000,
        15000,
        15000,
        15000,
        15000,
        15000
    ]
})

metas["mes"] = pd.PeriodIndex(metas["mes"], freq="M")

metas

,mes,meta
0,2024-01,15000
1,2024-02,15000
2,2024-03,15000
3,2024-04,15000
4,2024-05,15000
5,2024-06,15000


## 3.4 Comparación con las metas

In [12]:
resumen = resumen.reset_index()

resumen = resumen.merge(
    metas,
    on="mes",
    how="left"
)

resumen

,mes,total,numero_ventas,meta
0,2024-01,13773.50,10,15000
1,2024-02,13050.00,10,15000
2,2024-03,14454.00,11,15000
3,2024-04,16689.00,11,15000
4,2024-05,11922.25,9,15000
5,2024-06,11209.25,7,15000


## - Calculo Porcentaje de cumplimiento

In [13]:
resumen["cumplimiento"] = (
    resumen["total"] / resumen["meta"]
) * 100

resumen

,mes,total,numero_ventas,meta,cumplimiento
0,2024-01,13773.50,10,15000,91.823333
1,2024-02,13050.00,10,15000,87.000000
2,2024-03,14454.00,11,15000,96.360000
3,2024-04,16689.00,11,15000,111.260000
4,2024-05,11922.25,9,15000,79.481667
5,2024-06,11209.25,7,15000,74.728333


## 3.5 Exportar la información

In [15]:
# Se converierte la columna mes a texto
df["mes"] = df["mes"].astype(str)

# Se crea la base de datos
engine = create_engine("sqlite:///../output/techventas.db")

# Finalmente se debe Exportar
df.to_sql(
    "ventas",
    engine,
    if_exists="replace",
    index=False
)

58

## -Se verifica la información

In [16]:
pd.read_sql(
    "SELECT COUNT(*) FROM ventas",
    engine
)

,COUNT(*)
0,58


## Conclusión

Durante esta actividad se utilizó Pandas para preparar el conjunto de datos antes de su análisis. Se identificaron y eliminaron registros con información inconsistente, se crearon nuevas variables para calcular los ingresos y se realizó un resumen mensual de las ventas. Finalmente, los datos fueron exportados a una base de datos SQLite para facilitar su consulta en etapas posteriores.

Este ejercicio me permitió comprender la importancia de la limpieza y transformación de los datos, ya que una buena preparación ayuda a obtener resultados más confiables y facilita el análisis posterior.